# BENZI — LoRA fine-tune on Colab (~2–3 hours)

**Runtime:** Menu → *Runtime* → *Change runtime type* → **T4 GPU**

This notebook:
1. Clones your repo (or upload `fyp-ml-demos` if private)
2. Prepares empathy dataset
3. LoRA trains **Qwen2.5-3B** (4-bit GPU, ~60 steps)
4. Merges weights
5. Zips download for Ollama on your Mac

After download, on Mac see `benzi-server/docs/COLAB_TRAIN.md`.

In [ ]:
# Check GPU
!nvidia-smi
import torch
print('CUDA:', torch.cuda.is_available())
if not torch.cuda.is_available():
    raise RuntimeError('Enable GPU: Runtime → Change runtime type → T4 GPU')

In [ ]:
# Clone once into /content (re-running %cd nested paths — always use absolute path)
import os
import subprocess
from pathlib import Path

CONTENT = Path("/content")
REPO_DIR = CONTENT / "final-year-benzi"
ML_DIR = REPO_DIR / "fyp-ml-demos"

if not REPO_DIR.is_dir():
    subprocess.run(
        [
            "git", "clone", "--depth", "1",
            "https://github.com/sameedsaeed123/final-year-benzi.git",
            str(REPO_DIR),
        ],
        check=True,
    )

os.chdir(ML_DIR)
print("Working directory:", ML_DIR)
assert (ML_DIR / "finetune" / "train_qlora.py").is_file(), (
    "finetune/ missing on GitHub — push fyp-ml-demos/finetune or upload fyp-ml-demos zip"
)

**Private repo?** Skip clone. Use left folder 📁 → Upload `fyp-ml-demos` zip, then:
```python
%cd /content/fyp-ml-demos
```

In [ ]:
import subprocess
import sys
from pathlib import Path

def pip(*args):
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *args])

pip("-r", "requirements.txt")
# Colab: requirements-finetune-colab.txt — no numpy<2 (avoids jax/opencv warnings)
colab_req = Path("requirements-finetune-colab.txt")
if colab_req.is_file():
    pip("-r", str(colab_req))
else:
    pip(
        "datasets>=2.19.0", "peft>=0.11.0", "accelerate>=0.30.0",
        "sentencepiece>=0.2.0", "protobuf>=4.25.0", "tqdm>=4.66.0",
    )
pip("bitsandbytes>=0.43.0", "accelerate")

import numpy as np
import bitsandbytes as bnb
import torch
print("numpy", np.__version__, "| bitsandbytes", bnb.__version__, "| CUDA", torch.cuda.is_available())
if not torch.cuda.is_available():
    raise RuntimeError("Enable T4 GPU: Runtime → Change runtime type → GPU")


In [ ]:
# Step 1 — dataset (~5 min)
!python finetune/prepare_dataset.py --max-total 1200

In [ ]:
# Step 2 — train 3B LoRA on GPU (~45–90 min)
!python finetune/train_qlora.py --model Qwen/Qwen2.5-3B-Instruct --max-steps 60 --max-length 384 --batch-size 1 --grad-accum 4

In [ ]:
# Lighter / faster option (~20 min) — uncomment instead of cell above:
# !python finetune/train_qlora.py --benzi-lite

In [ ]:
# Step 3 — merge (~10 min)
!python finetune/merge_lora.py --model Qwen/Qwen2.5-3B-Instruct

In [ ]:
# Step 4 — zip for download
import shutil
from google.colab import files

shutil.make_archive('benzi-empathetic-trained', 'zip', 'finetune/merged/benzi-empathetic-hf')
shutil.make_archive('benzi-lora-adapter', 'zip', 'finetune/adapters/benzi-lora')
print('Downloading merged model + adapter zip…')
files.download('benzi-empathetic-trained.zip')
files.download('benzi-lora-adapter.zip')

## On your Mac (after download)

1. Unzip `benzi-empathetic-trained.zip` to e.g. `~/benzi-models/benzi-empathetic-hf`
2. Create Ollama model:
```bash
cd benzi-models/benzi-empathetic-hf
cat > Modelfile << 'EOF'
FROM .
PARAMETER temperature 0.65
PARAMETER num_ctx 4096
SYSTEM You are BENZI AI — supportive wellness between therapy sessions. Not a therapist. Defer clinical questions to their therapist.
EOF
ollama create benzi-empathetic-trained -f Modelfile
```
3. In `benzi-server/.env`: `OLLAMA_MODEL=benzi-empathetic-trained`
4. Restart API + keep `RAG_ENABLED=true` for context.